In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import joblib

housing = fetch_california_housing()
X, y = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

joblib.dump(model, '/content/house_price_model.joblib')
print("Model berhasil disimpan!")


Model berhasil disimpan!


In [2]:
%%writefile /content/predictor.py

import joblib
import numpy as np

def load_model(path="/content/house_price_model.joblib"):
    try:
        model = joblib.load(path)
        return model
    except FileNotFoundError:
        raise FileNotFoundError(f"File model tidak ditemukan di path: {path}")

def preprocess(data: dict):
    expected_features = [
        "MedInc", "HouseAge", "AveRooms", "AveBedrms",
        "Population", "AveOccup", "Latitude", "Longitude"
    ]

    missing = [f for f in expected_features if f not in data]
    if missing:
        raise ValueError(f"Fitur berikut tidak ditemukan: {missing}")

    for key in expected_features:
        if not isinstance(data[key], (int, float)):
            raise TypeError(f"Nilai '{key}' harus berupa angka, bukan {type(data[key]).__name__}")

    input_array = np.array([[data[f] for f in expected_features]])
    return input_array

def predict(data: dict):
    try:
        model = load_model()
        input_array = preprocess(data)
        result = model.predict(input_array)
        return {"predicted_price": round(float(result[0]) * 100000, 2)}
    except (ValueError, TypeError) as e:
        return {"error": str(e)}
    except Exception as e:
        return {"error": f"Terjadi kesalahan tidak terduga: {str(e)}"}

Writing /content/predictor.py


In [3]:
import importlib, sys

# Reload predictor
if 'predictor' in sys.modules:
    del sys.modules['predictor']

sys.path.insert(0, '/content')
from predictor import predict

sample_input = {
    "MedInc": 8.3252,
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.024,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23
}

result = predict(sample_input)
print("Hasil prediksi:", result)

Hasil prediksi: {'predicted_price': 426579.3}


In [4]:
bad_input = {
    "MedInc": "delapan",  # sengaja salah tipe
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.024,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23
}

result = predict(bad_input)
print("Hasil error handling:", result)

Hasil error handling: {'error': "Nilai 'MedInc' harus berupa angka, bukan str"}


In [5]:
!pip install fastapi uvicorn pyngrok -q

In [6]:
%%writefile /content/main.py

from fastapi import FastAPI
from pydantic import BaseModel
import sys
sys.path.insert(0, '/content')
from predictor import predict

app = FastAPI()

class HouseInput(BaseModel):
    MedInc: float
    HouseAge: float
    AveRooms: float
    AveBedrms: float
    Population: float
    AveOccup: float
    Latitude: float
    Longitude: float

@app.get("/")
def home():
    return {"message": "House Price Prediction API is running!"}

@app.post("/predict")
def predict_price(data: HouseInput):
    input_dict = data.dict()
    result = predict(input_dict)
    return result

Writing /content/main.py


In [7]:
import subprocess, time

time.sleep(2)

process = subprocess.Popen(
    ["npx", "localtunnel", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

for line in process.stdout:
    decoded = line.decode()
    print(decoded, end="")
    if "url" in decoded.lower():
        break

your url is: https://tiny-lands-peel.loca.lt


In [14]:
from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token("3EwIELOl3NRMBFGPUSmHNfMaglB_4sXryQC17CiWibJw9UcEm")
ngrok.kill()  # matikan semua tunnel lokal

def run():
    uvicorn.run("main:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run)
thread.daemon = True
thread.start()

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

INFO:     Started server process [1067]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Public URL: NgrokTunnel: "https://federal-custard-cruelty.ngrok-free.dev" -> "http://localhost:8000"


**TEST PREDIKSI NORMAL**

In [15]:
import requests

url = "https://federal-custard-cruelty.ngrok-free.dev/predict"

payload = {
    "MedInc": 8.3252,
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.024,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23
}

headers = {"ngrok-skip-browser-warning": "true"}

response = requests.post(url, json=payload, headers=headers)
print("Status code:", response.status_code)
print("Hasil prediksi:", response.json())

INFO:     34.57.229.189:0 - "POST /predict HTTP/1.1" 200 OK
Status code: 200
Hasil prediksi: {'predicted_price': 426579.3}


**TEST ERROR HANDLING VIA API**

In [16]:
bad_payload = {
    "MedInc": "delapan",  # sengaja salah tipe
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.024,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23
}

response = requests.post(url, json=bad_payload, headers=headers)
print("Status code:", response.status_code)
print("Hasil error handling:", response.json())

INFO:     34.57.229.189:0 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
Status code: 422
Hasil error handling: {'detail': [{'type': 'float_parsing', 'loc': ['body', 'MedInc'], 'msg': 'Input should be a valid number, unable to parse string as a number', 'input': 'delapan'}]}


**DOKUMENTASI**

In [21]:
%%writefile /content/README.md

# House Price Prediction API

> Praktikum Perancangan Aplikasi Sains Data — Kelompok 2 | Kelas DS-04-02

API berbasis FastAPI untuk memprediksi harga rumah menggunakan model Machine Learning (Random Forest Regressor) yang dilatih pada dataset California Housing dari scikit-learn.

---

## Anggota Kelompok

| No | Nama Lengkap | NIM |
|----|--------------|-----|
| 1 | Ni Luh Made Sri Utami Pradnyandari | 103102400055 |
| 2 | Safa Ayu Artanti | 103102430004 |

---

## Struktur File

TugasKelompokPASD/

├── predictor.py        # Modul load model, preprocessing, dan inferensi

├── main.py             # Web Service FastAPI (endpoint /predict)

├── deployment.ipynb    # Notebook Google Colab (training + deployment)

├── README.md           # Dokumentasi proyek

└── .gitignore          # Exclude file model (.joblib)

> ⚠️ File `house_price_model.joblib` tidak disertakan karena ukurannya melebihi batas GitHub (>100MB). Generate ulang dengan menjalankan Cell 1 di `deployment.ipynb`.

---

## Instalasi Dependencies

Jalankan perintah berikut untuk menginstall semua library yang dibutuhkan:

```bash
pip install fastapi uvicorn pyngrok scikit-learn joblib numpy
```

---

## Cara Menjalankan

### 1. Generate Model
Jalankan Cell 1 di `deployment.ipynb` untuk melatih dan menyimpan model:
```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import joblib

housing = fetch_california_housing()
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
joblib.dump(model, '/content/house_price_model.joblib')
```

### 2. Jalankan Server
```bash
uvicorn main:app --host 0.0.0.0 --port 8000
```

### 3. Akses Dokumentasi API
Buka browser dan akses: http://localhost:8000/docs

---

## Endpoint

### GET /
Mengecek apakah server berjalan.

**Response:**
```json
{"message": "House Price Prediction API is running!"}
```

### POST /predict
Menerima data rumah dalam format JSON dan mengembalikan prediksi harga.

---

## Deskripsi Fitur Input

| Fitur | Deskripsi | Tipe |
|-------|-----------|------|
| MedInc | Median pendapatan rumah tangga di blok tersebut | float |
| HouseAge | Usia rata-rata rumah di blok tersebut | float |
| AveRooms | Rata-rata jumlah ruangan per rumah tangga | float |
| AveBedrms | Rata-rata jumlah kamar tidur per rumah tangga | float |
| Population | Jumlah penduduk di blok tersebut | float |
| AveOccup | Rata-rata jumlah penghuni per rumah tangga | float |
| Latitude | Garis lintang lokasi blok rumah | float |
| Longitude | Garis bujur lokasi blok rumah | float |

---

## Contoh Payload (Input Valid)

```json
{
    "MedInc": 8.3252,
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.024,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23
}
```

---

## Contoh Response Sukses

```json
{
    "predicted_price": 426579.3
}
```

---

## Contoh Response Error (Input Salah Tipe)

```json
{
    "detail": [
        {
            "type": "float_parsing",
            "loc": ["body", "MedInc"],
            "msg": "Input should be a valid number, unable to parse string as a number",
            "input": "delapan"
        }
    ]
}
```

---

## Referensi

- [FastAPI Documentation](https://fastapi.tiangolo.com)
- [Scikit-learn California Housing Dataset](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset)
- [Repository GitHub](https://github.com/niluhmadesri/TugasKelompokPASD)

Overwriting /content/README.md


In [23]:
from google.colab import files

files.download('/content/house_price_model.joblib')
files.download('/content/predictor.py')
files.download('/content/main.py')
files.download('/content/README.md')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>